# MedPilot - Colab API Server

Host FastAPI tren Colab de local goi

**Runtime -> Change runtime type -> GPU**

In [ ]:
# 1. Cai dat
!pip install fastapi uvicorn transformers accelerate bitsandbytes pyngrok huggingface_hub -q

In [ ]:
# 2. Login HuggingFace
from huggingface_hub import login

HF_TOKEN = ""
# HF_TOKEN = "hf_xxx..."  # Đã xóa token thật, hãy nhập lại khi chạy notebook
# login(HF_TOKEN)
print("Login thanh cong!")

In [ ]:
# 3. Import
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from fastapi import FastAPI
from pyngrok import ngrok
import uvicorn
import time

In [ ]:
# 4. Setup ngrok (LAY TOKEN: https://dashboard.ngrok.com)
NGROK_TOKEN = "3BJuRXeiwgSNG5zWfm89ESHTDtg_3QBZ3CU3f7doQ3a6stpQS"
ngrok.set_auth_token(NGROK_TOKEN)

In [ ]:
# 5. Tai model
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

print("Dang tai tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Dang tai model (4-bit)...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config
)

print("Model tai xong!")

In [ ]:
# 6. Tao API
app = FastAPI(title="MedPilot LLM API")

def generate(prompt, max_tokens=200, temperature=0.7):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    
    outputs = model.generate(**inputs, max_new_tokens=max_tokens, temperature=temperature)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return response

@app.post("/v1/chat/completions")
async def chat_completions(data: dict):
    messages = data.get("messages", [])
    max_tokens = data.get("max_tokens", 200)
    temperature = data.get("temperature", 0.7)
    
    # Ghep tin nhan
    full_prompt = ""
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        full_prompt += f"{role.upper()}: {content}\n"
    full_prompt += "ASSISTANT: "
    
    response = generate(full_prompt, max_tokens, temperature)
    
    return {
        "choices": [{
            "message": {
                "role": "assistant",
                "content": response
            }
        }]
    }

@app.get("/health")
def health():
    return {"status": "ok"}

In [ ]:
# 7. Start server
print("=" * 60)
print("STARTING SERVER...")
print("=" * 60)

# Tao tunnel
public_url = ngrok.connect(8000)
print(f"\n>>> PUBLIC URL: {public_url}")

print(f"\n>>> COPY URL SAU VAO .env:")
print(f"VLLM_API_URL={public_url}/v1/chat/completions")

print("\n" + "=" * 60)
print("Server dang chay tren Colab!")
print("Giu tab nay mo!")
print("=" * 60)

# Chay server
uvicorn.run(app, host="0.0.0.0", port=8000)

---

## Huong dan:

1. **Chay tat ca o** theo thu tu 1 -> 7
2. **Doi** load model (~5-10 phut)
3. **Copy PUBLIC URL** tu o cuoi
4. **Paste vao .env** cua ban:
   ```
   VLLM_API_URL=https://xxx.ngrok.io/v1/chat/completions
   ```

5. **Chay backend local**

**Luu y:** Giu tab Colab mo de server chay!